<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 1</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">1.3</span>Volatilidad que cambia en el tiempo: GARCH</div>
</div>

**Objetivo.** modelar la volatilidad condicional y pronosticarla, en lugar de asumir que es constante.

**Lo que vamos a hacer**

- Observar los grupos de volatilidad en un activo colombiano.
- Ajustar un GARCH(1,1) con errores t.
- Pronosticar la volatilidad a 10 días.
- Probar el efecto apalancamiento con GJR-GARCH.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Datos

In [ ]:
prices = download_prices(["EC", "^GSPC"], start="2012-01-01")
asset = prices.columns[0]
r = 100 * to_returns(prices[asset])          # retornos logarítmicos en %
realized = r.rolling(21).std() * np.sqrt(252)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
r.plot(ax=ax[0], color="#5f5f5f", lw=.8, title=f"{asset} — retornos diarios (%)")
realized.plot(ax=ax[1], title="Volatilidad realizada a 21 días (anualizada, %)"); plt.show()

Se ven **grupos de volatilidad**: días turbulentos seguidos de más días turbulentos. GARCH(1,1) lo captura: $\sigma_t^2=\omega+\alpha\,\varepsilon_{t-1}^2+\beta\,\sigma_{t-1}^2$.

## 2. Ajuste

> ### ✏️ Ejercicio 1 — GARCH(1,1)
> Ajusta el modelo con `arch_model(r, vol="GARCH", p=1, q=1, dist="t")` y guarda el resultado en `res`.

In [ ]:
from arch import arch_model
# TODO: res = ...
res = None

In [ ]:
from arch import arch_model
am = arch_model(r, vol="GARCH", p=1, q=1, dist="t")
res = am.fit(disp="off")
print(res.params.round(4))

In [ ]:
fig, ax = plt.subplots()
(res.conditional_volatility * np.sqrt(252)).plot(ax=ax, label="GARCH(1,1)")
realized.plot(ax=ax, color="#5f5f5f", lw=1, label="Realizada 21d")
ax.set_title("Volatilidad condicional vs. realizada (anualizada, %)"); ax.legend(); plt.show()
print(f"Persistencia α+β = {res.params['alpha[1]'] + res.params['beta[1]']:.3f}")

## 3. Pronóstico

> ### ✏️ Ejercicio 2 — Pronóstico a 10 días
> Obtén la volatilidad anualizada pronosticada para los próximos 10 días (`res.forecast`).

In [ ]:
# TODO: vol10 = ...
vol10 = None

In [ ]:
fc = res.forecast(horizon=10)
vol10 = np.sqrt(fc.variance.iloc[-1] * 252)
print(vol10.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(1, 11), vol10.values, color="#A6192E")
ax.set_xlabel("Días hacia adelante"); ax.set_title("Volatilidad pronosticada (anualizada, %)"); plt.show()

## 4. Efecto apalancamiento

> ### ✏️ Ejercicio 3 — GJR-GARCH
> Añade `o=1` para que los retornos negativos aumenten más la volatilidad. Compara el AIC y mira el coeficiente `gamma[1]`.

In [ ]:
# TODO

In [ ]:
res2 = arch_model(r, vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
print(f"AIC  GARCH: {res.aic:,.1f} | GJR: {res2.aic:,.1f}  (menor es mejor)")
print(f"gamma[1] = {res2.params['gamma[1]']:.4f}  (>0: las caídas generan más volatilidad)")

## Para llevarse

- La volatilidad no es constante: se agrupa y vuelve lentamente a su nivel medio.
- GARCH da un pronóstico, no solo una medición del pasado, y alimenta VaR dinámicos.
- Con datos sintéticos (sin conexión) los resultados cambian, pero el procedimiento es el mismo.

---
*Excel para mirar, Python para repetir.*